<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/Torch_Tensorboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torchvision.transforms.v2 as transforms

from torch import nn
from torchvision import datasets
from torch.utils.tensorboard import SummaryWriter

In [ ]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
init_transforms = transforms.Compose(
    [transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)]
)

In [ ]:
train_set = datasets.MNIST(
    root="data", train=True, download=True, transform=init_transforms
)
test_set = datasets.MNIST(
    root="data", train=False, download=True, transform=init_transforms
)

In [ ]:
train_set = torch.utils.data.Subset(dataset=train_set, indices=range(1000))
test_set = torch.utils.data.Subset(test_set, range(1000))

In [ ]:
train_data = torch.stack([img for img, _ in train_set])
mean = train_data.mean().item()
std = train_data.std().item()

print(mean)
print(std)

0.1282389760017395
0.30513906478881836


In [ ]:
final_transform = transforms.Compose(
    [
        transforms.ToImage(), # array of pixel values between 0 and 255 (float64)
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize((mean,), (std,))
    ]
)


In [ ]:
train_set.transform = final_transform
test_set.transform = final_transform

In [ ]:
train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=32, shuffle=False)

In [ ]:
print(f"Train batch mean: {next(iter(train_loader))[0].mean()}")
print(f"Test batch mean: {next(iter(test_loader))[0].mean()}")

Train batch mean: 0.14894427359104156
Test batch mean: 0.11835766583681107


In [ ]:
print(f"Train batch std: {next(iter(train_loader))[0].std()}")
print(f"Test batch std: {next(iter(test_loader))[0].std()}")

Train batch std: 0.2999194860458374
Test batch std: 0.2935943603515625


In [ ]:
next(iter(train_loader))[0].shape

torch.Size([32, 1, 28, 28])

In [ ]:
class MyModel(nn.Module):

    def __init__(self):
        super().__init__() # initialize the parent class
        self.conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding="same")
        self.pool = nn.MaxPool2d(kernel_size=2) # shrink the image size by 2
        self.dropout = nn.Dropout(0.1)
        self.flatten = nn.Flatten()
        self.dense1 = nn.Linear(32 * int(28/2) * int(28/2), 64)
        self.dense2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.dropout(x)
        return self.dense2(x)

In [ ]:
torch.manual_seed(42)
model = MyModel()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

MyModel(
  (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.1, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dense1): Linear(in_features=6272, out_features=64, bias=True)
  (dense2): Linear(in_features=64, out_features=10, bias=True)
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def accuracy(outputs, y):
    pred = outputs.argmax(dim=1, keepdim=True) # in_dim = [batch_dim, 10], out_dim = [batch_dim, 1]
    correct = pred.eq(y.view_as(pred)).sum().item() # compute num of correct predictions
    return correct / pred.shape[0]

In [ ]:
EPOCHS = 5

# Create a tensorboard writer
os.makedirs("logs/mnist", exist_ok=True)
writer = SummaryWriter(log_dir="logs/mnist")

for epoch in range(EPOCHS):

    # Reset metrics
    train_loss = 0
    test_loss = 0
    train_acc = 0
    test_acc = 0

    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device) # move tensors to cpu/cuda
        optimizer.zero_grad()
        outputs = model(x)
        loss = loss_fn(outputs, y)
        loss.backward() # compute grads wrt to the loss
        optimizer.step() # apply grads

        train_loss += loss.item()
        train_acc += accuracy(outputs, y)

    train_loss /= len(train_loader) # compute mean loss over the train set
    train_acc /= len(train_loader)

    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = loss_fn(outputs, y)
            test_loss += loss.item()
            test_acc += accuracy(outputs, y)

    test_loss /= len(test_loader)
    test_acc /= len(test_loader)

    # Log to the terminal
    message = (
        f"Epoch {epoch}, Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f},"
        f"Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}"
    )

    print(message)

    # Log loss to TB
    writer.add_scalar("Loss/Train", train_loss, step=epoch)
    writer.add_scalar("Loss/Test", test_loss, step=epoch)
    writer.add_scalar("Acc/Train", train_acc, step=epoch)
    writer.add_scalar("Acc/Test", test_acc, step=epoch)

    # Log gradients
    for name, param in model.named_parameters():
        writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

    # Log norms of grads




